# 06 — MPC 設計者ジャーニー: 失敗・成功・体験（統合）

**目的:** Phase 1–4 の試行錯誤を **1 本の Notebook** で体験する。  
読み物: [MPC_TUNING_JOURNEY.md](../MPC_TUNING_JOURNEY.md) · 早見表: [TUNING_GUIDE.md](../TUNING_GUIDE.md)

> 各 Step は `scripts/tuning_labs.py` の **lab ID** と 1:1 対応。CLI でも同じ実験が再現できます。


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

from tuning_labs import (
    TUNING_LABS,
    list_labs,
    run_lab,
    run_lab_pair,
    plot_speed_trial_journey,
    plot_param_study_mu,
    load_cached_lab_results,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 0 — Lab カタログ

`tuning_labs.py` に登録された fail / success ペア一覧。


In [ ]:
for lab in list_labs():
    print(f"{lab.id:28} [{lab.phase:7}] session={lab.session}  {lab.title}")
print(f"\n{len(TUNING_LABS)} labs total")


## Phase 1 — ref_z（MPC 以前の物理パラメータ）

[Phase 1 — 平坦スモーク](../MPC_TUNING_JOURNEY.md#phase-1-flat-smoke)


In [ ]:
from pympc_lab import compare_runs
pair = run_lab_pair("s1_ref_z_fail", "s1_ref_z_ok")
fig = compare_runs(pair)
plt.suptitle("Phase 1: ref_z fail vs ok", y=1.02)
plt.show()
for label, m in pair:
    print(label, "terminated=", m["terminated"], "min_z=", round(m["min_z"], 3))


## Phase 2 — μ と step_freq（平坦）

[Phase 2 — 平坦チューニング](../MPC_TUNING_JOURNEY.md#phase-2-flat-tune)


In [ ]:
pair_mu = run_lab_pair("s2_mu_aggressive", "s2_mu_conservative")
fig = compare_runs(pair_mu)
plt.suptitle("Phase 2a: mu aggressive vs conservative", y=1.02)
plt.show()

fail_freq = run_lab("s2_step_freq_fast")
ok_freq = run_lab("s2_mu_conservative")  # baseline gait for contrast
fig = compare_runs([
    ("step_freq=1.6 FAIL", fail_freq["metrics"]),
    ("mu=0.35 stable gait", ok_freq["metrics"]),
])
plt.suptitle("Phase 2b: step_freq too fast", y=1.02)
plt.show()


In [ ]:
# 事前計測 param study（scripts/run_parameter_study.py）
fig = plot_param_study_mu()
plt.show()


## Phase 3 — 不整地（boxes / perlin）

[Phase 3 — 不整地](../MPC_TUNING_JOURNEY.md#phase-3-rough-terrain)

| デモ | GIF |
|------|-----|
| boxes | ![boxes](../assets/demo_s03_boxes.gif) |
| perlin | ![perlin](../assets/demo_s03_perlin.gif) |


In [ ]:
pair_boxes = run_lab_pair("s3_boxes_freq_fail", "s3_boxes_freq_ok")
fig = compare_runs(pair_boxes)
plt.suptitle("Phase 3a: boxes step_freq fail vs ok", y=1.02)
plt.show()

fail_perlin = run_lab("s3_perlin_mu_fail")
print("perlin mu=0.55:", fail_perlin["result"]["terminated"], fail_perlin["result"]["distance_m"])


## Phase 4 — 5 kph × 凸凹坂

[Phase 4 — 5 kph × 凸凹坂](../MPC_TUNING_JOURNEY.md#phase-4-speed-bumpy)

| 地形 | GIF | meta |
|------|-----|------|
| flat | ![](../assets/demo_s04_flat.gif) | demo_s04_flat.meta.json |
| uphill | ![](../assets/demo_s04_uphill.gif) | demo_s04_uphill.meta.json |
| downhill | ![](../assets/demo_s04_downhill.gif) | demo_s04_downhill.meta.json |


In [ ]:
import json
# 全試行ログの可視化
fig = plot_speed_trial_journey()
plt.show()

fail_s4 = run_lab("s4_no_fall_fail")
print("no-fall 5kph:", fail_s4["result"])


In [ ]:
# ✅ resilient 勝ちパラメータ（時間がかかる — 1 地形だけ実行例）
# 全 3 地形は Notebook 05 または tuning_labs.py --lab s4_resilient_* で

win = run_lab("s4_resilient_flat_win")
print(win["result"]["distance_m"], "m", "falls=", win["result"].get("falls"))
fig, ax = plt.subplots(figsize=(9, 3))
m = win["metrics"]
ax.plot(m["x"], m["vx"] * 3.6, lw=1.5)
ax.axhline(5.0, ls="--", color="k", alpha=0.4)
ax.set_xlabel("cumulative distance [m]")
ax.set_ylabel("vx [kph]")
ax.set_title("Phase 4 success: bumpy_flat resilient 20m")
ax.grid(True, alpha=0.3)
plt.show()


## Step 7 — あなたの手で 1 パラメータ変更

`run_lab("s4_resilient_flat_win")` の kwargs を **1 つだけ** 変えて再実行し、距離・転倒を記録。

例: `duty_factor` を 0.70 に下げると falls が増えるか？


In [ ]:
# --- 演習: 下の kwargs を 1 つだけ編集 ---
from pympc_lab import apply_preset, run_speed_terrain_sim_resilient

apply_preset("session04_speed_bumpy_base")
custom = run_speed_terrain_sim_resilient(
    scene="bumpy_flat",
    target_speed_kph=5.0,
    min_distance_m=20.0,
    max_seconds=90.0,
    max_falls=22,
    mu=0.42,
    step_freq=1.20,
    duty_factor=0.76,   # ← ここを 0.70 などに変更して試す
    ref_z_scale=1.07,
    speed_ramp_s=18.0,
)
print("distance_m", custom["distance_m"], "falls", custom["falls"], "success", custom["success"])


## Step 8 — 修了チェック

- [ ] Phase 1–4 各 1 つの **fail lab** と **success lab** を実行した
- [ ] [MPC_TUNING_JOURNEY.md](../MPC_TUNING_JOURNEY.md) のトリアージフローを説明できる
- [ ] `python scripts/verify_workshop_assets.py` でデモ meta を確認した
- [ ] 勝ちパラメータが `configs/pympc_presets/session04_bumpy_*.yaml` に保存されていることを確認した

**CLI 再現:**
```bash
python scripts/tuning_labs.py --list
python scripts/tuning_labs.py --lab s2_mu_aggressive
python scripts/verify_workshop_assets.py
```
